# 10 Oracle Action Pipeline + BioBART


## 1. Imports and Configuration

The pipeline uses gold labels directly:

- `rephrase` -> BioBART
- `split` -> BioBART
- `ignore` -> original complex sentence
- `delete` -> empty string

`merge` and `none` are excluded from evaluation.

In [ ]:
from __future__ import annotations

import ast
import gc
import random
import re
from pathlib import Path
from typing import Any

import evaluate
import numpy as np
import pandas as pd
import sacrebleu
import torch
from bert_score import score as bert_score
from tqdm.auto import tqdm
from transformers import AutoModelForSeq2SeqLM, AutoTokenizer

SEED = 42
MODEL_NAME = "GanjinZero/biobart-v2-base"
ALLOWED_LABELS = ["rephrase", "delete", "ignore", "split"]
GENERATION_LABELS = ["rephrase", "split"]

GENERATION_CONFIG = {
    "max_new_tokens": 128,
    "num_beams": 4,
    "length_penalty": 0.9,
    "no_repeat_ngram_size": 3,
    "early_stopping": True,
}

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

RAW_CANDIDATES = {
    "train": [
        PROJECT_ROOT / "data" / "raw" / "cochraneauto_sents_train.csv",
        PROJECT_ROOT / "data" / "sentence" / "raw" / "cochraneauto_sents_train.csv",
    ],
    "validation": [
        PROJECT_ROOT / "data" / "raw" / "cochraneauto_sents_val.csv",
        PROJECT_ROOT / "data" / "sentence" / "raw" / "cochraneauto_sents_val.csv",
    ],
    "test": [
        PROJECT_ROOT / "data" / "raw" / "cochraneauto_sents_test.csv",
        PROJECT_ROOT / "data" / "sentence" / "raw" / "cochraneauto_sents_test.csv",
    ],
}

PROCESSED_CANDIDATES = {
    "train": [
        PROJECT_ROOT / "data" / "sentence_no_context" / "train_clean.csv",
        PROJECT_ROOT / "data" / "sentence_no_context" / "processed" / "train.csv",
    ],
    "validation": [
        PROJECT_ROOT / "data" / "sentence_no_context" / "val_clean.csv",
        PROJECT_ROOT / "data" / "sentence_no_context" / "processed" / "val.csv",
    ],
    "test": [
        PROJECT_ROOT / "data" / "sentence_no_context" / "test_clean.csv",
        PROJECT_ROOT / "data" / "sentence_no_context" / "processed" / "test.csv",
    ],
}

CHECKPOINT_CANDIDATES = [
    PROJECT_ROOT / "models" / "biobart_sentence_no_context" / "best_model",
    PROJECT_ROOT / "models" / "biobart_sentence_no_context",
    PROJECT_ROOT / "models" / "biobart_knowledge_enhanced" / "best_model",
    PROJECT_ROOT / "models" / "biobart_knowledge_enhanced",
]

RESULTS_DIR = PROJECT_ROOT / "results"
PREDICTION_PATH = RESULTS_DIR / "oracle_action_pipeline_biobart_predictions.csv"
METRICS_PATH = RESULTS_DIR / "oracle_action_pipeline_biobart_metrics.csv"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

pd.set_option("display.max_colwidth", 180)

print(f"Project root: {PROJECT_ROOT}")
print(f"Allowed labels: {ALLOWED_LABELS}")
print(f"Generation config: {GENERATION_CONFIG}")
print(f"CUDA available: {torch.cuda.is_available()}")

## 2. Load Labeled Sentence Data



In [ ]:
def first_existing(paths: list[Path]) -> Path | None:
    return next((path for path in paths if path.exists()), None)

def parse_simple_value(value: Any) -> str:
    if pd.isna(value):
        return ""
    if isinstance(value, list):
        return " ".join(str(item).strip() for item in value if str(item).strip())
    text = str(value).strip()
    if text.startswith("[") and text.endswith("]"):
        try:
            parsed = ast.literal_eval(text)
            if isinstance(parsed, list):
                return " ".join(str(item).strip() for item in parsed if str(item).strip())
        except (ValueError, SyntaxError):
            pass
    return text

def load_labeled_split(split_name: str) -> pd.DataFrame:
    raw_path = first_existing(RAW_CANDIDATES[split_name])
    processed_path = first_existing(PROCESSED_CANDIDATES[split_name])
    path = raw_path or processed_path
    if path is None:
        raise FileNotFoundError(f"No file found for {split_name}")

    df = pd.read_csv(path)
    required = ["complex", "simple", "label"]
    missing = [column for column in required if column not in df.columns]
    if missing:
        raise ValueError(f"{path} is missing columns: {missing}")

    if "pair_id" not in df.columns:
        df["pair_id"] = np.arange(len(df)).astype(str)
    if "sent_id" not in df.columns:
        df["sent_id"] = np.arange(len(df))

    keep_columns = ["pair_id", "sent_id", "label", "complex", "simple"]
    df = df[keep_columns].copy()
    df["label"] = df["label"].fillna("").astype(str).str.strip().str.lower()
    df["complex"] = df["complex"].fillna("").astype(str).str.strip()
    df["simple"] = df["simple"].map(parse_simple_value).fillna("").astype(str).str.strip()
    df = df[df["complex"].ne("")].reset_index(drop=True)
    df["source_file"] = str(path.relative_to(PROJECT_ROOT))
    return df

train_df = load_labeled_split("train")
val_df = load_labeled_split("validation")
test_df = load_labeled_split("test")

for split_name, df in [("train", train_df), ("validation", val_df), ("test", test_df)]:
    print(f"{split_name}: {len(df):,} rows from {df['source_file'].iloc[0]}")
    display(df["label"].value_counts(dropna=False).rename_axis("label").reset_index(name="count"))

display(test_df.head())

## 3. Filter Oracle Pipeline Labels


In [ ]:
def filter_allowed_labels(df: pd.DataFrame, split_name: str) -> pd.DataFrame:
    before = len(df)
    filtered = df[df["label"].isin(ALLOWED_LABELS)].copy().reset_index(drop=True)
    removed = before - len(filtered)
    print(f"{split_name}: kept {len(filtered):,} / {before:,}; removed {removed:,} merge/none/other examples")
    display(filtered["label"].value_counts().rename_axis("label").reset_index(name="count"))
    return filtered

train_oracle_df = filter_allowed_labels(train_df, "train")
val_oracle_df = filter_allowed_labels(val_df, "validation")
test_oracle_df = filter_allowed_labels(test_df, "test")

## 4. Load Fine-Tuned BioBART


In [ ]:
def has_model_files(path: Path) -> bool:
    return path.exists() and (path / "config.json").exists()

local_checkpoint = next((path for path in CHECKPOINT_CANDIDATES if has_model_files(path)), None)
model_source = str(local_checkpoint) if local_checkpoint is not None else MODEL_NAME
tokenizer_source = model_source if local_checkpoint is not None and (Path(model_source) / "tokenizer_config.json").exists() else MODEL_NAME

if local_checkpoint is None:
    print("WARNING: No local fine-tuned BioBART checkpoint found. Falling back to pretrained base BioBART.")
else:
    print(f"Loading fine-tuned BioBART checkpoint: {Path(model_source).relative_to(PROJECT_ROOT)}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(tokenizer_source)
model = AutoModelForSeq2SeqLM.from_pretrained(model_source)
model.to(device)
model.eval()

print(f"Tokenizer source: {tokenizer_source}")
print(f"Model source: {model_source}")
print(f"Device: {device}")

## 5. Oracle Action Pipeline


In [ ]:
PROMPT_TEMPLATE = """You are an expert in biomedical text simplification.

Rewrite the biomedical sentence for a general audience.

Rules:
- Preserve the original meaning.
- Use clear and simple language.
- Replace medical, scientific, or technical terms with simpler alternatives whenever possible.
- Remove unnecessary statistical details unless they are essential for understanding the main finding.
- Do not add information that is not present in the original sentence.
- Output exactly one simplified sentence.

Sentence:
{complex_sentence}

Simplified sentence:"""

def build_biobart_input(sentence: str) -> str:
    return PROMPT_TEMPLATE.format(complex_sentence=str(sentence).strip())

def clean_prediction(text: str) -> str:
    """Remove prompt echoes and generation boilerplate from decoded text."""
    text = re.sub(r"\s+", " ", str(text).strip())
    if not text:
        return ""

    # Decoder-only or poorly adapted seq2seq checkpoints can echo the input prompt.
    prompt_markers = [
        "Simplified sentence:",
        "Rewrite this biomedical sentence in simpler language:",
        "Simplify the biomedical sentence.",
        "Sentence:",
    ]
    for marker in prompt_markers:
        if marker in text:
            text = text.split(marker)[-1].strip()

    prefixes = ["Simplified:", "Answer:", "Prediction:"]
    for prefix in prefixes:
        if text.lower().startswith(prefix.lower()):
            text = text[len(prefix):].strip()

    return re.sub(r"\s+", " ", text).strip()

def generate_biobart(sentences: list[str], batch_size: int = 8) -> list[str]:
    predictions: list[str] = []
    inputs = [build_biobart_input(sentence) for sentence in sentences]
    for start in tqdm(range(0, len(inputs), batch_size), desc="BioBART generation"):
        batch_inputs = inputs[start:start + batch_size]
        try:
            encoded = tokenizer(
                batch_inputs,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=512,
            ).to(device)
            with torch.no_grad():
                generated = model.generate(**encoded, **GENERATION_CONFIG)
            decoded = tokenizer.batch_decode(generated, skip_special_tokens=True)
            predictions.extend([clean_prediction(text) for text in decoded])
        except Exception as exc:
            print(f"Generation failed for batch starting at {start}: {exc}")
            predictions.extend([""] * len(batch_inputs))
        finally:
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
            gc.collect()
    return predictions

def apply_oracle_pipeline(df: pd.DataFrame, batch_size: int = 8) -> pd.DataFrame:
    output = df.copy()
    output["oracle_action"] = output["label"]
    output["prediction"] = ""

    ignore_mask = output["label"].eq("ignore")
    delete_mask = output["label"].eq("delete")
    generation_mask = output["label"].isin(GENERATION_LABELS)

    output.loc[ignore_mask, "prediction"] = output.loc[ignore_mask, "complex"]
    output.loc[delete_mask, "prediction"] = ""

    generation_indices = output.index[generation_mask].tolist()
    generation_sentences = output.loc[generation_indices, "complex"].tolist()
    generation_predictions = generate_biobart(generation_sentences, batch_size=batch_size)
    output.loc[generation_indices, "prediction"] = generation_predictions

    return output

test_prediction_df = apply_oracle_pipeline(test_oracle_df, batch_size=8)
test_prediction_df.to_csv(PREDICTION_PATH, index=False)
print(f"Saved predictions to: {PREDICTION_PATH.relative_to(PROJECT_ROOT)}")
display(test_prediction_df[["pair_id", "sent_id", "label", "complex", "simple", "oracle_action", "prediction"]].head())



## 6. Evaluation Functions


In [ ]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.evaluation import compute_metrics

def compute_text_metrics(df: pd.DataFrame) -> pd.DataFrame:
    """Use the shared text simplification metrics."""
    return compute_metrics(df)


def metric_value(summary: pd.DataFrame, metric: str) -> float:
    values = summary.loc[summary["metric"].eq(metric), "score"].tolist()
    return float(values[0]) if values else float("nan")


def delete_accuracy(df: pd.DataFrame) -> float:
    delete_df = df[df["label"].eq("delete")].copy()
    if len(delete_df) == 0:
        return float("nan")
    return float(delete_df["prediction"].fillna("").astype(str).str.strip().eq("").mean())


## 7. Overall and Per-Label Evaluation



In [ ]:
evaluation_rows = []

for subset_name, subset_df in [("all oracle pipeline examples", test_prediction_df)]:
    metrics = compute_text_metrics(subset_df)
    evaluation_rows.append({
        "subset": subset_name,
        "rows": len(subset_df),
        "metric_examples": metric_value(metrics, "metric_examples"),
        "SARI": metric_value(metrics, "SARI"),
        "BLEU": metric_value(metrics, "BLEU"),
        "BERTScore Precision": metric_value(metrics, "BERTScore Precision"),
        "BERTScore Recall": metric_value(metrics, "BERTScore Recall"),
        "BERTScore F1": metric_value(metrics, "BERTScore F1"),
        "delete_accuracy": delete_accuracy(subset_df),
    })

for label in ALLOWED_LABELS:
    subset_df = test_prediction_df[test_prediction_df["label"].eq(label)].copy()
    metrics = compute_text_metrics(subset_df)
    evaluation_rows.append({
        "subset": f"{label} only",
        "rows": len(subset_df),
        "metric_examples": metric_value(metrics, "metric_examples"),
        "SARI": metric_value(metrics, "SARI"),
        "BLEU": metric_value(metrics, "BLEU"),
        "BERTScore Precision": metric_value(metrics, "BERTScore Precision"),
        "BERTScore Recall": metric_value(metrics, "BERTScore Recall"),
        "BERTScore F1": metric_value(metrics, "BERTScore F1"),
        "delete_accuracy": delete_accuracy(subset_df),
    })

metrics_df = pd.DataFrame(evaluation_rows)
metrics_df.to_csv(METRICS_PATH, index=False)
print(f"Saved metrics to: {METRICS_PATH.relative_to(PROJECT_ROOT)}")
display(metrics_df)

## 8. Comparison Table



In [ ]:
overall = metrics_df[metrics_df["subset"].eq("all oracle pipeline examples")].iloc[0]
comparison_df = pd.DataFrame([
    {"Experiment": "E2", "Model": "BioBART direct", "SARI": 31.86, "BLEU": 30.91, "BERTScore F1": 0.932},
    {"Experiment": "E4", "Model": "classifier + BioBART", "SARI": 27.01, "BLEU": 22.20, "BERTScore F1": 0.932},
    {
        "Experiment": "E8",
        "Model": "oracle label + BioBART",
        "SARI": overall["SARI"],
        "BLEU": overall["BLEU"],
        "BERTScore F1": overall["BERTScore F1"],
    },
])
display(comparison_df)

## 9. Analysis



In [ ]:
for label in ALLOWED_LABELS:
    subset = test_prediction_df[test_prediction_df["label"].eq(label)].copy()
    if len(subset) == 0:
        continue
    print(f"Label: {label}")
    display(subset[["complex", "simple", "oracle_action", "prediction"]].sample(n=min(10, len(subset)), random_state=SEED))